In [21]:
# Bibliotecas e configurações gerais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import xlrd
from datetime import datetime

# Somente para reprodutibilidade
np.random.seed(42)

# Configurações visuais
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (14,5)
plt.rcParams["figure.dpi"] = 100

# Caminhos
RAW     = Path("../data/raw")
PROC    = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)

print("Setup configurado")

Setup configurado


In [22]:
# Célula 2: Leitura dos arquivos
def ler_cepea(caminho):
    # Abre ignorando a corrupção reportada pelo xlrd
    wb = xlrd.open_workbook(
        caminho,
        ignore_workbook_corruption=True
    )
    # Pega a primeira aba
    sheet = wb.sheet_by_index(0)

    # Converte para DataFrame
    dados = []
    for i in range(sheet.nrows):
        dados.append(sheet.row_values(i))

    df = pd.DataFrame(dados)
    # Primeira linha vira cabeçalho
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)
    return df

# Lendo os três arquivos
df_boi   = ler_cepea(RAW / "cepea-consulta-boigordo-20260418230100.xls")
df_milho = ler_cepea(RAW / "cepea-consulta-milho-20260418225845.xls")
df_soja  = ler_cepea(RAW / "cepea-consulta-soja-20260418230026.xls")

# Diagnóstico rápido
for nome, df in [("Boi Gordo", df_boi), ("Milho", df_milho), ("Soja", df_soja)]:
    print(f"\n{'='*80}")
    print(f"{nome}")
    print(f"Shape: {df.shape}")
    print(f"Colunas: {df.columns.tolist()}")
    print(df.head(3))


Boi Gordo
Shape: (1560, 2)
Colunas: ['Boi | INDICADOR DO BOI GORDO CEPEA/ESALQ', '']
0 Boi | INDICADOR DO BOI GORDO CEPEA/ESALQ  \
0                                     Nota   
1                                    Fonte   
2                                     Data   

0                                                     
0  por arroba, descontado o Prazo de Pagamento pe...  
1                                              Cepea  
2                                              Valor  

Milho
Shape: (1561, 3)
Colunas: ['Milho | INDICADOR DO MILHO ESALQ/BM&FBOVESPA', '', '']
0 Milho | INDICADOR DO MILHO ESALQ/BM&FBOVESPA  \
0                                         Nota   
1                                        Fonte   
2                                         Data   

0                                                                  
0  à vista por saca de 60 kg, descontado o Prazo ...               
1                                              Cepea               
2             

In [25]:
# Célula 3: Limpeza e padronização dos dados CEPEA
def ler_cepea_limpo(caminho, nome_produto):
    wb = xlrd.open_workbook(caminho, ignore_workbook_corruption=True)
    sheet = wb.sheet_by_index(0)

    dados = []
    for i in range(sheet.nrows):
        dados.append(sheet.row_values(i))

    df = pd.DataFrame(dados)

    # Linha 2 = cabeçalho real (Data, Valor...)
    df.columns = df.iloc[2]
    df = df[3:].reset_index(drop=True)

    # Renomeia pra padronizar
    df = df.rename(columns={
        df.columns[0]: 'data',
        df.columns[1]: 'preco_rs'
    })

    # Remover linhas vazias
    df = df[df['data'].notna() & (df['data'] != '')]
    df = df[df['preco_rs'].notna() & (df['preco_rs'] != '')]

    # Converte data
    def converter_data(valor):
        try:
            return datetime(*xlrd.xldate_as_tuple(float(valor), wb.datemode))
        except:
            return pd.NaT

    df['data'] = df['data'].apply(converter_data)
    df['preco_rs'] = pd.to_numeric(df['preco_rs'], errors='coerce')

    # Adiciona coluna de produto
    df['produto'] = nome_produto

    df = df.dropna().reset_index(drop=True)

    return df

# Lendo arquivos formatados
df_boi   = ler_cepea_limpo(RAW / "cepea-consulta-boigordo-20260418230100.xls", "boi_gordo")
df_milho = ler_cepea_limpo(RAW / "cepea-consulta-milho-20260418225845.xls",    "milho")
df_soja  = ler_cepea_limpo(RAW / "cepea-consulta-soja-20260418230026.xls",     "soja")

for nome, df in [("Boi Gordo", df_boi), ("Milho", df_milho), ("Soja", df_soja)]:
    print(f"\n{'='*80}")
    print(f"{nome}")
    print(f"Shape: {df.shape}")
    print(f"Período: {df['data'].min()} → {df['data'].max()}")
    print(df.head(3))


Boi Gordo
Shape: (0, 3)
Período: NaT → NaT
Empty DataFrame
Columns: [data, preco_rs, produto]
Index: []

Milho
Shape: (0, 4)
Período: NaT → NaT
Empty DataFrame
Columns: [data, preco_rs, À vista US$, produto]
Index: []

Soja
Shape: (0, 4)
Período: NaT → NaT
Empty DataFrame
Columns: [data, preco_rs, À vista US$, produto]
Index: []


In [24]:
# Célula diagnóstico — ver todas as linhas brutas do arquivo
wb = xlrd.open_workbook(
    RAW / "cepea-consulta-boigordo-20260418230100.xls",
    ignore_workbook_corruption=True
)
sheet = wb.sheet_by_index(0)

print(f"Total de linhas: {sheet.nrows}")
print(f"Total de colunas: {sheet.ncols}")
print("\n--- Primeiras 10 linhas brutas ---")
for i in range(min(10, sheet.nrows)):
    print(f"Linha {i}: {sheet.row_values(i)}")

Total de linhas: 1561
Total de colunas: 2

--- Primeiras 10 linhas brutas ---
Linha 0: ['Boi | INDICADOR DO BOI GORDO CEPEA/ESALQ', '']
Linha 1: ['Nota', 'por arroba, descontado o Prazo de Pagamento pela taxa CDI/CETIP']
Linha 2: ['Fonte', 'Cepea']
Linha 3: ['Data', 'Valor']
Linha 4: ['02/01/2020', '192,95']
Linha 5: ['03/01/2020', '196,70']
Linha 6: ['06/01/2020', '196,40']
Linha 7: ['07/01/2020', '196,60']
Linha 8: ['08/01/2020', '196,70']
Linha 9: ['09/01/2020', '198,45']
